# 005 - Function Calling

This notebook demonstrates how to enable LLMs to call external functions, extending their capabilities beyond text generation.

## Setup

Import libraries and configure the API.

In [1]:
import os
from datetime import datetime

import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API key from environment variable
api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please check your .env file.")

# Configure the API
genai.configure(api_key=api_key)
model = genai.GenerativeModel("models/gemini-2.0-flash-lite")

## Define Functions

Create the functions that the LLM will be able to call.

In [2]:
def fibonacci(n):
    """Calculate the nth Fibonacci number.

    Args:
        n (int): The position in the Fibonacci sequence (must be >= 0)

    Returns:
        int: The nth Fibonacci number
    """
    if n < 0:
        raise ValueError("n must be non-negative")
    elif n <= 1:
        return n
    else:
        a, b = 0, 1
        for _ in range(2, n + 1):
            a, b = b, a + b
        return b


def get_current_time():
    """Get the current date and time.

    Returns:
        str: Current date and time in a readable format
    """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

## Define Function Declarations

Create function declarations that tell the LLM about available functions.

In [3]:
# Define function declarations for the LLM
fibonacci_declaration = genai.protos.FunctionDeclaration(
    name="fibonacci",
    description="Calculate the nth Fibonacci number",
    parameters=genai.protos.Schema(
        type=genai.protos.Type.OBJECT,
        properties={
            "n": genai.protos.Schema(
                type=genai.protos.Type.INTEGER,
                description="The position in the Fibonacci sequence (must be >= 0)"
            )
        },
        required=["n"]
    )
)

get_time_declaration = genai.protos.FunctionDeclaration(
    name="get_current_time",
    description="Get the current date and time",
    parameters=genai.protos.Schema(
        type=genai.protos.Type.OBJECT,
        properties={}
    )
)

# Create a tool with our functions
math_tool = genai.protos.Tool(
    function_declarations=[fibonacci_declaration, get_time_declaration]
)

# Function mapping for execution
available_functions = {
    "fibonacci": fibonacci,
    "get_current_time": get_current_time
}

## Function Execution Helper

Create a helper to execute function calls from the LLM.

In [15]:
def call_function(function_call):
    """Execute a function call and return the result"""
    function_name = function_call.name
    function_args = {}

    # Extract arguments and convert types as needed
    for key, value in function_call.args.items():
        # Convert float to int for integer parameters
        if isinstance(value, float) and value.is_integer():
            function_args[key] = int(value)
        else:
            function_args[key] = value

    print(f"🔧 Calling: {function_name}({function_args})")

    func = available_functions[function_name]
    result = func(**function_args)
    print(f"✅ Result: {result}")
    return result

## Main Function Calling Interface

Create the main function that enables the LLM to call functions.

In [16]:
def ask_with_functions(prompt):
    """Ask the LLM a question with function calling enabled"""
    print(f"\n🎯 Query: {prompt}")
    print("-" * 50)

    system_prompt = f"""You are a specialized assistant that can help with mathematical calculations and time-related queries.

CAPABILITIES:
- Calculate Fibonacci numbers for any non-negative integer
- Provide current date and time information

INSTRUCTIONS:
1. If the user asks about Fibonacci numbers or mathematical sequences, use the fibonacci function
2. If the user asks about current time, date, or "what time is it", use the get_current_time function
3. For any other topics outside these capabilities, politely decline and redirect to your available functions

RESPONSE GUIDELINES:
- Be helpful and precise
- Always use the appropriate function when the query matches your capabilities
- If you cannot help, explain what you CAN do instead
- Keep responses concise but informative

USER QUERY: {prompt}"""

    response = model.generate_content(system_prompt, tools=[math_tool])

    if response.candidates[0].content.parts:
        for part in response.candidates[0].content.parts:
            if hasattr(part, 'function_call') and part.function_call:
                call_function(part.function_call)
            elif hasattr(part, 'text') and part.text:
                print(f"💬 Response: {part.text}")

## Example 1: Fibonacci Calculation

Ask the LLM to calculate a Fibonacci number.

In [17]:
ask_with_functions("What is the 6th Fibonacci number?")


🎯 Query: What is the 6th Fibonacci number?
--------------------------------------------------
🔧 Calling: fibonacci({'n': 6})
✅ Result: 8


## Example 2: Current Time

Ask the LLM for the current time.

In [18]:
ask_with_functions("What time is it right now?")


🎯 Query: What time is it right now?
--------------------------------------------------
🔧 Calling: get_current_time({})
✅ Result: 2025-10-20 10:53:15


## Example 3: Non-valid questoin

Ask the LLM for something it was not tasked to do.

In [19]:
ask_with_functions("What is the capital of Kenya?")


🎯 Query: What is the capital of Kenya?
--------------------------------------------------
💬 Response: I am sorry, I am unable to answer questions about the capital of Kenya. However, I can help you with Fibonacci number calculations and provide the current date and time.



## Try Your Own Queries

Use the cell below to test your own function calling queries.

In [ ]:
# Try your own queries here
# ask_with_functions("Your question here")